In [ ]:
# Installation des bibliothèques Python
!pip install streamlit pennylane datasets scikit-learn pandas matplotlib seaborn -q

# Pas besoin d'installer localtunnel via pip !
# On vérifiera juste que Node.js est présent (il l'est par défaut sur Colab)
!node -v

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 48.8 MB/s eta 0:00:00
v20.19.0


In [ ]:
%%writefile app.py
import streamlit as st
import pennylane as qml
from pennylane import numpy as np
import pandas as pd
import time
import tracemalloc
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.decomposition import PCA
from datasets import load_dataset

# ============================================================
# CONFIG
# ============================================================
st.set_page_config(page_title="Quantum Benchmark ", layout="wide")
st.markdown("""
<style>
.method-card {
    background: linear-gradient(135deg, #0d1b2a 0%, #1b263b 100%);
    border-left: 4px solid #4a90d9;
    border-radius: 10px;
    padding: 2rem;
    margin-top: 1rem;
    margin-bottom: 1.5rem;
    color: #e0e0e0;
    font-size: 0.95rem;
    line-height: 1.8;
}
.method-card h3 { color: #7eb8f7; margin-bottom: 0.8rem; font-size: 1.3rem; }
.method-card code {
    background: #0a0f1e;
    color: #a8d8ea;
    padding: 2px 6px;
    border-radius: 4px;
    font-size: 0.88rem;
}
.tag {
    display: inline-block;
    background: #1f4068;
    color: #90caf9;
    border-radius: 4px;
    padding: 2px 10px;
    font-size: 0.85rem;
    margin: 2px;
}
.section-title {
    background: linear-gradient(90deg, #1565c0 0%, #6a1b9a 100%);
    color: white;
    padding: 0.6rem 1.2rem;
    border-radius: 8px;
    font-size: 1.1rem;
    font-weight: bold;
    margin: 1.5rem 0 1rem 0;
}
.stButton>button {
    background: linear-gradient(90deg, #1565c0, #6a1b9a);
    color: white;
    border: none;
    border-radius: 8px;
    font-weight: bold;
    padding: 0.6rem 2rem;
}
</style>
""", unsafe_allow_html=True)

st.title("🌌 Quantum Benchmark")
st.markdown("**Exploration et comparaison des méthodes d'encodage quantique** — Dataset : Pima Indians Diabetes")

# ============================================================
# DÉFINITIONS
# ============================================================
METHOD_DEFINITIONS = {
    "Raw Data (Baseline)": {
        "description": "Référence classique sans encodage quantique. Les données normalisées dans [0,1] par min-max scaling sont directement transmises au classifieur. Sert de référence absolue pour quantifier l'apport de chaque encodage quantique.",
        "formule": "X_out = X_norm  (identité, pas de transformation quantique)",
        "qubits": "0 qubit — pipeline entièrement classique",
        "avantage": "Temps d'exécution minimal, référence comparative absolue.",
        "limite": "Aucun enrichissement de l'espace de features.",
        "couleur": "#b2bec3", "icone": "⚫"
    },
    "Angle Encoding": {
        "description": "Chaque feature classique xᵢ ∈ [0,1] est convertie en angle de rotation appliqué à un qubit dédié via RX et RY. L'état final est le produit tensoriel des états individuels. La partie réelle du vecteur d'état (dim 256) est extraite comme features.",
        "formule": "RX(xᵢ·π) · RY(xᵢ·π/2) |0⟩ → Re(|ψ⟩) ∈ ℝ²⁵⁶",
        "qubits": "8 qubits — 1 qubit par feature",
        "avantage": "Intuitif, continu, préserve la structure géométrique.",
        "limite": "Pas d'intrication — features traitées indépendamment.",
        "couleur": "#4a90d9", "icone": "🔵"
    },
    "Phase Encoding": {
        "description": "Encode les features dans la phase complexe des amplitudes. Une porte Hadamard crée une superposition uniforme, puis RZ encode la feature dans la phase : (|0⟩ + e^{ixᵢπ}|1⟩)/√2. L'information est accessible via des circuits d'interférence.",
        "formule": "|0⟩ →[H]→ →[RZ(xᵢπ)]→ (|0⟩ + e^{ixᵢπ}|1⟩)/√2",
        "qubits": "8 qubits — 1 qubit par feature",
        "avantage": "Encode dans la phase, exploitable via l'interférence quantique.",
        "limite": "Phase indétectable sans circuit d'interférence dédié.",
        "couleur": "#7b2ff7", "icone": "🟣"
    },
    "Amplitude Encoding": {
        "description": "Encode un vecteur de N features directement dans les amplitudes d'un état quantique à ⌈log₂N⌉ qubits. Pour 8 features, seulement 3 qubits suffisent. Le vecteur doit être de norme L2 unitaire. Compression exponentielle de l'information.",
        "formule": "|ψ⟩ = Σᵢ xᵢ|i⟩ avec ‖x‖₂=1 → 3 qubits pour 8 features",
        "qubits": "3 qubits — compression exponentielle (log₂(8)=3)",
        "avantage": "Compression exponentielle : N features → log₂(N) qubits.",
        "limite": "Normalisation L2 obligatoire ; préparation d'état coûteuse.",
        "couleur": "#00b894", "icone": "🟢"
    },
    "Density / Hybrid Encoding": {
        "description": "Représente l'état quantique par sa matrice densité ρ = |ψ⟩⟨ψ|. Cette représentation capture les cohérences entre états (termes hors-diagonaux ρᵢⱼ), invisibles dans la distribution de probabilité classique. La matrice est aplatie en vecteur de dim 64.",
        "formule": "ρ = |ψ⟩⟨ψ| = x_norm·x_normᵀ ∈ ℝ⁸ˣ⁸ → flatten → ℝ⁶⁴",
        "qubits": "8 qubits (conceptuel) — implémentation classique via matrice densité",
        "avantage": "Capture les corrélations croisées entre features.",
        "limite": "Dimension de sortie N² croît quadratiquement.",
        "couleur": "#e17055", "icone": "🟠"
    },
    "Feature Map Encoding": {
        "description": "Projette les données dans un espace de Hilbert via un circuit IQP (Instantaneous Quantum Polynomial), analogue aux noyaux kernel SVM. Une PCA à 3 composantes précède l'encodage. Le noyau quantique K(x,z)=|⟨φ(x)|φ(z)⟩|² est potentiellement intractable classiquement.",
        "formule": "H⊗³ → RZ(π·xᵢ) → CZ → RY(π·xᵢ) | K(x,z)=|⟨φ(x)|φ(z)⟩|²",
        "qubits": "3 qubits (après PCA à 3 composantes)",
        "avantage": "Kernel quantique puissant pour données non linéairement séparables.",
        "limite": "PCA entraîne une perte d'information ; sensible aux répétitions.",
        "couleur": "#fdcb6e", "icone": "🟡"
    },
    "Variational Encoding": {
        "description": "Data re-uploading : les features sont ré-injectées dans le circuit, intercalées avec des paramètres entraînables θ. Deux couches alternent RY(xᵢπ)·RZ(θᵢ) avec intrication CNOT. Approximateur universel quantique selon Pérez-Salinas et al. (2020).",
        "formule": "[RY(xᵢπ)·RZ(θᵢ) + CNOT] × 2 couches → |ψ|² ∈ ℝ²⁵⁶",
        "qubits": "8 qubits — 16 paramètres entraînables (2 couches × 8 qubits)",
        "avantage": "Expressivité maximale via re-uploading ; approximateur universel.",
        "limite": "Paramètres non optimisés dans ce benchmark (initialisation aléatoire).",
        "couleur": "#ff7675", "icone": "🔴"
    },
    "Block Encoding": {
        "description": "Encode la matrice diagonale des features dans un bloc d'un opérateur unitaire plus grand, via des qubits ancilla. Inspiré des algorithmes HHL et QLSA. Des rotations contrôlées ctrl-RY appliquent la feature uniquement si tous les ancilla sont dans |1⟩.",
        "formule": "H⊗³(ancilla) → ctrl-RY(xᵢ) → ⟨Z_{ancilla}⟩ ∈ ℝ¹",
        "qubits": "11 qubits — 3 ancilla + 8 data qubits",
        "avantage": "Fondement des algorithmes quantiques d'algèbre linéaire.",
        "limite": "Sortie scalaire (dim 1) — très faible expressivité pour la classification.",
        "couleur": "#636e72", "icone": "🔲"
    },
    "Directional Encoding": {
        "description": "Exploite les trois degrés de liberté de chaque qubit sur la sphère de Bloch via RX·RY·RZ avec la même valeur xᵢ. La non-commutativité des rotations crée un encodage riche. Une couche CNOT en chaîne crée ensuite de l'intrication entre qubits voisins.",
        "formule": "RX(xᵢ)·RY(xᵢ)·RZ(xᵢ)|0⟩ + CNOT chaîne → |ψ|² ∈ ℝ²⁵⁶",
        "qubits": "8 qubits — 3 rotations par qubit + CNOT en chaîne",
        "avantage": "Exploite les 3 axes de la sphère de Bloch ; intrication entre voisins.",
        "limite": "Redondance : 3 rotations avec la même valeur xᵢ.",
        "couleur": "#00cec9", "icone": "🔷"
    },
    "Entangler Enhanced": {
        "description": "Inspiré de l'évolution temporelle sous un Hamiltonien de champ transverse. Étape 1 : superposition globale H⊗⁸. Étape 2 : évolution de phase RZ(xᵢ). Étape 3 : intrication CNOT en chaîne. L'information de chaque feature est distribuée globalement via l'intrication.",
        "formule": "H⊗⁸ → RZ(xᵢ) → CNOT chaîne → |ψ|² ∈ ℝ²⁵⁶",
        "qubits": "8 qubits — évolution temporelle + intrication globale",
        "avantage": "Représentation collective des features via l'intrication.",
        "limite": "Information individuelle diluée dans l'état global.",
        "couleur": "#6c5ce7", "icone": "🔮"
    },
    "QSample Encoding": {
        "description": "Encode les features comme probabilités de mesure : P(|1⟩) = xᵢ. La porte RY(2·arcsin(√xᵢ)) prépare chaque qubit tel que sa probabilité de mesure soit exactement xᵢ. La valeur d'espérance ⟨Zᵢ⟩ = 1-2xᵢ ∈ [-1,1] est extraite comme feature.",
        "formule": "RY(2·arcsin(√xᵢ))|0⟩ → ⟨Zᵢ⟩ = 1-2xᵢ ∈ [-1,1]⁸",
        "qubits": "8 qubits — encodage probabiliste via arcsin",
        "avantage": "Encodage probabiliste naturel ; sortie dans [-1,1] interprétable.",
        "limite": "Transformation monotone décroissante — peut inverser l'ordre des features.",
        "couleur": "#a29bfe", "icone": "🫧"
    },
    "Chebyshev Encoding": {
        "description": "Motivé par la théorie de l'approximation fonctionnelle. Les polynômes de Chebyshev Tₙ(x)=cos(n·arccos(x)) forment une base orthogonale optimale. θᵢ=arccos(2xᵢ-1), puis RY(θᵢ) encode T₁ et RY(2θᵢ) encode T₂, projetant implicitement dans un espace polynomial de degré 2.",
        "formule": "θᵢ=arccos(2xᵢ-1) → RY(θᵢ)·RY(2θᵢ)|0⟩ → probs ∈ ℝ²⁵⁶",
        "qubits": "8 qubits — encodage via polynômes T₁ et T₂ de Chebyshev",
        "avantage": "Capture des relations quadratiques ; base optimale pour l'approximation.",
        "limite": "Limité aux degrés 1 et 2 dans cette implémentation.",
        "couleur": "#55efc4", "icone": "🟦"
    },
    "Fourier Encoding": {
        "description": "Encode deux harmoniques de Fourier par qubit : H → RZ(xᵢπ) → RZ(2xᵢπ). Fondé sur le cadre des quantum Fourier features (Schuld et al.) : les circuits quantiques paramétriques calculent naturellement des séries de Fourier tronquées, capturant des patterns périodiques.",
        "formule": "H → RZ(xᵢπ) → RZ(2xᵢπ) |0⟩ → probs ∈ ℝ²⁵⁶",
        "qubits": "8 qubits — 2 harmoniques de Fourier par qubit",
        "avantage": "Capture des patterns périodiques ; fondé sur les quantum Fourier features.",
        "limite": "Limité à 2 harmoniques ; pas d'intrication entre qubits.",
        "couleur": "#fab1a0", "icone": "🎵"
    },
    "Projected Unitary Encoding": {
        "description": "Mesure multi-observable : après AngleEmbedding(xᵢπ), les valeurs d'espérance ⟨Xᵢ⟩=sin(xᵢπ) et ⟨Zᵢ⟩=cos(xᵢπ) sont extraites pour chaque qubit. Les 16 features résultantes représentent chaque feature dans deux bases complémentaires avec la contrainte ⟨X⟩²+⟨Z⟩²=1.",
        "formule": "AngleEmb(xᵢπ) → [⟨Xᵢ⟩, ⟨Zᵢ⟩] pour i=1..8 → ℝ¹⁶",
        "qubits": "8 qubits — double projection sur observables X et Z",
        "avantage": "Double projection complémentaire ; sortie compacte de dimension 16.",
        "limite": "⟨X⟩²+⟨Z⟩²=1 — les 16 features ne sont pas indépendantes.",
        "couleur": "#74b9ff", "icone": "📐"
    },
    "Scaled Encoding": {
        "description": "Variante de l'Angle Encoding utilisant l'angle 2π au lieu de π, exploitant le cycle complet de la sphère de Bloch. Adapté aux données périodiques. Attention : RY(0)|0⟩ = RY(2π)|0⟩ = |0⟩ crée une ambiguïté aux bornes (xᵢ=0 et xᵢ=1 donnent le même état).",
        "formule": "RY(xᵢ·2π)|0⟩ → cos(xᵢπ)|0⟩+sin(xᵢπ)|1⟩ → probs ∈ ℝ²⁵⁶",
        "qubits": "8 qubits — cycle complet [0, 2π] de la sphère de Bloch",
        "avantage": "Exploite le cycle complet ; adapté aux données périodiques.",
        "limite": "Ambiguïté aux bornes : xᵢ=0 et xᵢ=1 donnent le même état.",
        "couleur": "#e84393", "icone": "📏"
    },
}

# ============================================================
# CHARGEMENT DES DONNÉES
# ============================================================
@st.cache_data
def load_data():
    ds = load_dataset("Genius-Society/Pima")
    df = pd.DataFrame(ds['train'])
    feature_cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
                    'Insulin','BMI','DiabetesPedigreeFunction','Age']
    X = df[feature_cols].values
    y = df['Outcome'].values
    X_norm = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
    return X_norm, y

X_all, y_all = load_data()
N_TOTAL = len(X_all)

dev8 = qml.device("default.qubit", wires=8)
dev3 = qml.device("default.qubit", wires=3)

# ============================================================
# ENCODAGES
# ============================================================
def run_encoding(X, method):
    if method == "Raw Data (Baseline)":
        return X
    elif method == "Angle Encoding":
        @qml.qnode(dev8)
        def circ(x):
            for i in range(8):
                qml.RX(x[i] * np.pi, wires=i)
                qml.RY(x[i] * (np.pi/2), wires=i)
            return qml.state()
        return np.real(np.array([circ(r) for r in X]))
    elif method == "Phase Encoding":
        @qml.qnode(dev8)
        def circ(x):
            for i in range(8):
                qml.Hadamard(wires=i)
                qml.RZ(x[i] * np.pi, wires=i)
            return qml.probs(wires=range(8))
        return np.array([circ(r) for r in X])
    elif method == "Amplitude Encoding":
        @qml.qnode(dev3)
        def circ(x):
            qml.AmplitudeEmbedding(x, wires=range(3), normalize=True, pad_with=0.0)
            return qml.state()
        return np.real(np.array([circ(r) for r in X]))
    elif method == "Density / Hybrid Encoding":
        def density(x):
            xn = x / (np.linalg.norm(x) + 1e-9)
            psi = xn.reshape(-1, 1)
            rho = np.dot(psi, psi.conj().T)
            return rho.real.flatten()
        return np.array([density(r) for r in X])
    elif method == "Feature Map Encoding":
        Xr = PCA(n_components=3).fit_transform(X)
        @qml.qnode(dev3)
        def circ(x):
            for i in range(3):
                qml.Hadamard(wires=i)
            for i in range(3):
                qml.RZ(np.pi * float(x[i]), wires=i)
            for i in range(2):
                qml.CZ(wires=[i, i+1])
            for i in range(3):
                qml.RY(np.pi * float(x[i]), wires=i)
            return qml.state()
        return np.real(np.array([circ(r) for r in Xr]))
    elif method == "Variational Encoding":
        np.random.seed(42)
        theta = np.random.rand(16)
        @qml.qnode(dev8)
        def circ(x, th):
            for i in range(8):
                qml.RY(x[i] * np.pi, wires=i)
                qml.RZ(th[i], wires=i)
            for i in range(7):
                qml.CNOT(wires=[i, i+1])
            for i in range(8):
                qml.RY(x[i] * np.pi, wires=i)
                qml.RZ(th[i+8], wires=i)
            return qml.state()
        return np.abs(np.array([circ(r, theta) for r in X]))**2
    elif method == "Block Encoding":
        n_data, n_anc = 8, 3
        dev_b = qml.device("default.qubit", wires=n_data+n_anc)
        @qml.qnode(dev_b)
        def circ(x):
            xn = x / (np.linalg.norm(x) + 1e-9)
            for i in range(n_anc):
                qml.Hadamard(wires=i)
            for i in range(n_data):
                qml.ctrl(qml.RY, control=range(n_anc))(xn[i], wires=n_anc+i)
            return qml.expval(qml.PauliZ(wires=n_anc))
        return np.array([[circ(r)] for r in X])
    elif method == "Directional Encoding":
        @qml.qnode(dev8)
        def circ(x):
            xn = x / (np.max(x) + 1e-9)
            for i in range(8):
                qml.RX(xn[i], wires=i)
                qml.RY(xn[i], wires=i)
                qml.RZ(xn[i], wires=i)
            for i in range(7):
                qml.CNOT(wires=[i, i+1])
            return qml.state()
        return np.abs(np.array([circ(r) for r in X]))**2
    elif method == "Entangler Enhanced":
        @qml.qnode(dev8)
        def circ(x):
            xn = x / (np.max(x) + 1e-9)
            for i in range(8):
                qml.Hadamard(wires=i)
            for i in range(8):
                qml.RZ(xn[i], wires=i)
            for i in range(7):
                qml.CNOT(wires=[i, i+1])
            return qml.state()
        return np.abs(np.array([circ(r) for r in X]))**2
    elif method == "QSample Encoding":
        @qml.qnode(dev8)
        def circ(x):
            for i in range(8):
                qml.RY(2*np.arcsin(np.sqrt(np.clip(x[i], 0, 1))), wires=i)
            return [qml.expval(qml.PauliZ(i)) for i in range(8)]
        return np.array([circ(r) for r in X])
    elif method == "Chebyshev Encoding":
        @qml.qnode(dev8)
        def circ(x):
            for i in range(8):
                angle = np.arccos(np.clip(2*x[i]-1, -1, 1))
                qml.RY(angle, wires=i)
                qml.RY(2*angle, wires=i)
            return qml.probs(wires=range(8))
        return np.array([circ(r) for r in X])
    elif method == "Fourier Encoding":
        @qml.qnode(dev8)
        def circ(x):
            for i in range(8):
                qml.Hadamard(wires=i)
                qml.RZ(x[i] * np.pi, wires=i)
                qml.RZ(2 * x[i] * np.pi, wires=i)
            return qml.probs(wires=range(8))
        return np.array([circ(r) for r in X])
    elif method == "Projected Unitary Encoding":
        @qml.qnode(dev8)
        def circ(x):
            qml.AngleEmbedding(x * np.pi, wires=range(8))
            return (
                [qml.expval(qml.PauliX(i)) for i in range(8)] +
                [qml.expval(qml.PauliZ(i)) for i in range(8)]
            )
        return np.array([circ(r) for r in X])
    elif method == "Scaled Encoding":
        @qml.qnode(dev8)
        def circ(x):
            for i in range(8):
                qml.RY(x[i] * 2 * np.pi, wires=i)
            return qml.probs(wires=range(8))
        return np.array([circ(r) for r in X])
    return X

# ============================================================
# SIDEBAR
# ============================================================
st.sidebar.header("⚙️ Configuration")
all_methods = list(METHOD_DEFINITIONS.keys())

selected = st.sidebar.multiselect(
    "Méthodes à benchmarker",
    all_methods,
    default=["Raw Data (Baseline)", "Angle Encoding", "Amplitude Encoding"]
)

limit = st.sidebar.slider(
    "Nombre d'échantillons",
    min_value=50, max_value=N_TOTAL, value=200, step=50,
    help=f"Dataset complet = {N_TOTAL} échantillons"
)

st.sidebar.markdown("---")
st.sidebar.markdown(f"**Dataset :** Pima Indians Diabetes")
st.sidebar.markdown(f"**Échantillons :** {limit} / {N_TOTAL}")
st.sidebar.markdown(f"**Classifieur :** Decision Tree (max_depth=5)")
st.sidebar.markdown(f"**Split :** 80% train / 20% test")

# ============================================================
# SECTION 1 — DÉFINITION INTERACTIVE (PLEINE LARGEUR)
# ============================================================
st.markdown('<div class="section-title">📖 Section 1 — Explorer une méthode d\'encodage</div>', unsafe_allow_html=True)

method_preview = st.selectbox(
    "Sélectionnez une méthode pour afficher sa définition complète :",
    all_methods
)

info = METHOD_DEFINITIONS[method_preview]
st.markdown(f"""
<div class="method-card">
<h3>{info['icone']} {method_preview}</h3>
<p>{info['description']}</p>
<hr style="border-color:#2d3e55; margin:1rem 0"/>
<p><b>📐 Formule :</b><br><code>{info['formule']}</code></p>
<p><b>💻 Qubits :</b> <span class="tag">{info['qubits']}</span></p>
<p><b>✅ Avantage :</b> {info['avantage']}</p>
<p><b>⚠️ Limite :</b> {info['limite']}</p>
</div>
""", unsafe_allow_html=True)

st.markdown("---")

# ============================================================
# SECTION 2 — BENCHMARK
# ============================================================
st.markdown('<div class="section-title">🚀 Section 2 — Lancer le Benchmark</div>', unsafe_allow_html=True)

if not selected:
    st.warning("⚠️ Sélectionnez au moins une méthode dans la barre latérale.")
else:
    st.info(f"**{len(selected)} méthode(s)** sélectionnée(s) sur **{limit}** échantillons")

    if st.button("▶ Lancer le Benchmark", use_container_width=True):
        X_sub = X_all[:limit]
        y_sub = y_all[:limit]
        results = []
        prog = st.progress(0, text="Initialisation...")

        for idx, method in enumerate(selected):
            prog.progress(idx / len(selected),
                          text=f"⏳ {method}  ({idx+1}/{len(selected)})")
            tracemalloc.start()
            t_start = time.time()
            X_enc = run_encoding(X_sub, method)
            X_train, X_test, y_train, y_test = train_test_split(
                X_enc, y_sub, test_size=0.2, random_state=42)
            clf = DecisionTreeClassifier(max_depth=5, random_state=42)
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            t_total = round(time.time() - t_start, 2)
            _, mem_peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()
            results.append({
                "Méthode":      method,
                "Accuracy (%)": round(accuracy_score(y_test, y_pred) * 100, 2),
                "Temps (s)":    t_total,
                "Mémoire (MB)": round(mem_peak / 1024**2, 3),
                "Qubits":       METHOD_DEFINITIONS[method]["qubits"],
                "CM":           confusion_matrix(y_test, y_pred),
                "y_test":       y_test,
                "y_pred":       y_pred,
            })

        prog.progress(1.0, text="✅ Benchmark terminé !")

        df_res = pd.DataFrame([
            {k: v for k, v in r.items() if k not in ("CM","y_test","y_pred")}
            for r in results
        ]).sort_values("Accuracy (%)", ascending=False).reset_index(drop=True)

        # ── TABLEAU ──────────────────────────────────────────
        st.markdown("### 📊 Tableau des résultats")
        st.dataframe(df_res, use_container_width=True)

        best = df_res.iloc[0]
        st.success(f"🏆 **Meilleure méthode :** {best['Méthode']} — **{best['Accuracy (%)']}%**")
        st.info(
            f"⏱ **Plus rapide :** {df_res.nsmallest(1,'Temps (s)')['Méthode'].values[0]}  |  "
            f"💾 **Moins de mémoire :** {df_res.nsmallest(1,'Mémoire (MB)')['Méthode'].values[0]}"
        )

        # ── GRAPHIQUES ───────────────────────────────────────
        st.markdown("### 📈 Visualisations comparatives")
        tab1, tab2, tab3, tab4 = st.tabs(
            ["🎯 Accuracy", "⏱ Temps d'exécution", "🧠 Mémoire", "📉 Matrices de Confusion"]
        )

        colors = [METHOD_DEFINITIONS[m]["couleur"] for m in df_res["Méthode"]]

        def styled_fig(figsize=(11, max(4, len(df_res)*0.6))):
            fig, ax = plt.subplots(figsize=figsize)
            fig.patch.set_facecolor('#0d1b2a')
            ax.set_facecolor('#0d1b2a')
            ax.tick_params(colors='white')
            for spine in ax.spines.values():
                spine.set_edgecolor('#2d3e55')
            return fig, ax

        with tab1:
            fig, ax = styled_fig()
            bars = ax.barh(df_res["Méthode"], df_res["Accuracy (%)"],
                           color=colors, edgecolor='white', linewidth=0.3)
            ax.set_xlabel("Accuracy (%)", color='white')
            ax.set_title("Accuracy par méthode d'encodage", color='white', fontsize=13)
            ax.set_xlim(0, 112)
            for bar, val in zip(bars, df_res["Accuracy (%)"]):
                ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
                        f"{val:.1f}%", va='center', color='white', fontsize=9)
            ax.invert_yaxis()
            plt.tight_layout()
            st.pyplot(fig)

        with tab2:
            fig, ax = styled_fig()
            ax.barh(df_res["Méthode"], df_res["Temps (s)"],
                    color=colors, edgecolor='white', linewidth=0.3)
            ax.set_xlabel("Temps (secondes)", color='white')
            ax.set_title("Temps d'exécution par méthode", color='white', fontsize=13)
            ax.invert_yaxis()
            plt.tight_layout()
            st.pyplot(fig)

        with tab3:
            fig, ax = styled_fig()
            ax.barh(df_res["Méthode"], df_res["Mémoire (MB)"],
                    color=colors, edgecolor='white', linewidth=0.3)
            ax.set_xlabel("Mémoire pic (MB)", color='white')
            ax.set_title("Consommation mémoire par méthode", color='white', fontsize=13)
            ax.invert_yaxis()
            plt.tight_layout()
            st.pyplot(fig)

        with tab4:
            cols = st.columns(2)
            for i, r in enumerate(results):
                with cols[i % 2]:
                    fig, ax = plt.subplots(figsize=(4, 3))
                    fig.patch.set_facecolor('#0d1b2a')
                    ax.set_facecolor('#0d1b2a')
                    ConfusionMatrixDisplay(
                        confusion_matrix=r["CM"],
                        display_labels=["Non-diab.", "Diab."]
                    ).plot(ax=ax, colorbar=False, cmap="Blues")
                    ax.set_title(
                        f"{r['Méthode']}\nAcc: {r['Accuracy (%)']}%",
                        color='white', fontsize=9)
                    ax.tick_params(colors='white')
                    for spine in ax.spines.values():
                        spine.set_edgecolor('#2d3e55')
                    plt.tight_layout()
                    st.pyplot(fig)

Writing app.py


In [ ]:
!fuser -k 8501/tcp

In [ ]:
!nohup streamlit run app.py &

nohup: appending output to 'nohup.out'


In [ ]:
!netstat -tuln | grep 8501

In [ ]:
!ssh -o StrictHostKeyChecking=no -R 80:localhost:8501 serveo.net

Forwarding HTTP traffic from https://b21d85a02713d7d2-136-118-53-213.serveousercontent.com
Tip (1): Create an account to reserve names. Pro removes the warning page: https://console.serveo.net/settings?n=1&src=ssh_nudge&v=A
Connection to serveo.net closed by remote host.
Connection to serveo.net closed.
